# Task 03 — Image preprocessing

**Wave 1** · target file: `backend/features.py` · prerequisites: none

**🎯 Goal:** `preprocess_image(arr, crop)` — turn a raw cutout into a model-ready array.

Part of the *2026-06-17 fused photo-z backend*. Plan & deps: `../PLAN.md` · conventions: `../README.md`.

In [ ]:
# === Setup: run me first. Hops to the repo root so `import backend` and models/ resolve,
# and pulls the model artifact(s) from GCS if missing (needs gcloud auth - see README). ===
import os, sys
p = os.path.abspath('.')
while p != '/' and not (os.path.isdir(p+'/backend') and os.path.isdir(p+'/notebooks')):
    p = os.path.dirname(p)
os.chdir(p); sys.path.insert(0, p)
print('repo root:', p)
os.makedirs('models', exist_ok=True)
if not os.path.exists('models/fake_image_model.pkl'):
    os.system('gcloud storage cp gs://macrocosm-lewagon/models/fake_image_model.pkl models/fake_image_model.pkl')
print('models:', [f for f in os.listdir('models') if f.endswith('.pkl')])

## What to build
`preprocess_image(arr, crop=None)` in `backend/features.py`.
- Accept a single `(64,64,5)` cutout **or** a batch `(n,64,64,5)`; output `(n,S,S,5) float32`.
- **center-crop** to `S = crop or settings.CROP`: `o=(64-S)//2; arr[:, o:o+S, o:o+S, :]`.
- **arcsinh stretch** (`np.arcsinh`) to compress the wide flux range.
- **per-image, per-channel normalize** (stateless, option (a) in the KB): for each image & channel
  subtract the mean and divide by the std (`+1e-6`).
- **raise `ValueError`** if a cutout isn't `(...,64,64,5)`.

## 🛠️ Develop & test here first
Fill the `# TODO` so the asserts pass. **Don't** paste the answer — write it.

In [ ]:
import numpy as np
IMG_SHAPE = (64, 64, 5)
def preprocess_image(arr, crop=64):
    # TODO: validate shape (ValueError on bad), center-crop to `crop`, arcsinh, per-image per-channel normalize
    raise NotImplementedError

a = preprocess_image(np.random.rand(64, 64, 5).astype("float32"))
assert a.shape == (1, 64, 64, 5)
assert abs(a[0, :, :, 0].mean()) < 1e-4 and abs(a[0, :, :, 0].std() - 1) < 1e-2   # normalized
assert preprocess_image(np.random.rand(3, 64, 64, 5).astype("float32")).shape == (3, 64, 64, 5)
try:
    preprocess_image(np.zeros((32, 32, 5))); print("FAIL: should raise")
except ValueError:
    print("OK: bad shape raises")

## ➡️ Move it into `backend/features.py`
Once the cell above passes, put your implementation into **`backend/features.py`** (replace the `# TODO`). Then run the **Check** cell — it imports from `backend/` and verifies the real module.

## ✅ Check (imports from `backend/`)

In [ ]:
import importlib, numpy as np, backend.features as F; importlib.reload(F)
a = F.preprocess_image(np.random.rand(64, 64, 5).astype("float32"))
assert a.shape[1:] == (F.settings.CROP, F.settings.CROP, 5)
print("preprocess_image OK", a.shape)

> ⚠️ **02 + 03 both edit `backend/features.py`.** Branch off a fresh `2026.6.17` and keep *both* functions when merging — see `MERGE.md`.

## 🚀 Ship it

In [ ]:
# === Commit & push on YOUR OWN branch (run last; repo root after Setup) ===
!git checkout -B task/03-features-image
!git add backend/features.py notebooks/tasks-2026-6-17/03-features-image/task.ipynb
!git commit -m "03-features-image: Image preprocessing"
!git push -u origin task/03-features-image
# then merge back into 2026.6.17 -> see MERGE.md in this folder